In [2]:
# =============================================================================
# DATATHON 2026 — Sales Forecasting Pipeline (Only sales.csv)
# Strategy: Prophet decomposition → components as features → LGBM/XGB/CatBoost
#           OOF Stacking (TimeSeriesSplit) → no leakage → SHAP explainability
# =============================================================================
 
# %% ── [0] Imports ────────────────────────────────────────────────────────────
import copy, os, warnings
warnings.filterwarnings('ignore')
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import shap
from prophet import Prophet
 
from lightgbm  import LGBMRegressor
import xgboost as xgb
from catboost  import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

SEED     = 42
DATA_DIR = "../data"          # ← thư mục chứa sales.csv & sample_submission.csv
OUT_DIR  = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

In [3]:
# %% ── [1] Load data ──────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 1 — Loading data")
print("=" * 65)
 
df  = pd.read_csv(os.path.join(DATA_DIR,'raw' ,"sales.csv"),
                  parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sub = pd.read_csv(os.path.join(DATA_DIR, 'raw',"sample_submission.csv"),
                  parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
 
print(f"Train : {df['Date'].min().date()} → {df['Date'].max().date()}  ({len(df):,} rows)")
print(f"Test  : {sub['Date'].min().date()} → {sub['Date'].max().date()}  ({len(sub):,} rows)")
 

STEP 1 — Loading data
Train : 2012-07-04 → 2022-12-31  (3,833 rows)
Test  : 2023-01-01 → 2024-07-01  (548 rows)


In [4]:
# %% ── [2] Vietnam holiday calendar ──────────────────────────────────────────
# Đây là lịch cố định (kiến thức chung), không phải external data
print("\nSTEP 2 — Building Vietnam holiday calendar")
 
# Tết Nguyên Đán (ngày mùng 1 âm lịch, convert sang dương lịch)
TET = {
    2013:"2013-02-10", 2014:"2014-01-31", 2015:"2015-02-19",
    2016:"2016-02-08", 2017:"2017-01-28", 2018:"2018-02-16",
    2019:"2019-02-05", 2020:"2020-01-25", 2021:"2021-02-12",
    2022:"2022-02-01", 2023:"2023-01-22", 2024:"2024-02-10",
}
 
def build_prophet_holidays():
    """Tạo DataFrame holidays cho Prophet."""
    rows = []
    # Tết — window rộng vì ảnh hưởng revenue mạnh (giảm mạnh khi nghỉ, spike trước Tết)
    for year, s in TET.items():
        tet = pd.Timestamp(s)
        rows.append(dict(holiday="tet_pre_spike",
                         ds=tet - pd.Timedelta(days=7),
                         lower_window=0, upper_window=6))   # tuần trước Tết: mua sắm tăng vọt
        rows.append(dict(holiday="tet_break",
                         ds=tet,
                         lower_window=0, upper_window=6))   # tuần nghỉ Tết: doanh thu giảm
    # Ngày lễ dương lịch
    fixed = {
        "New_Year":          "01-01",
        "Reunification_Day": "04-30",
        "Labor_Day":         "05-01",
        "National_Day":      "09-02",
        "Sale_1111":         "11-11",  # Ngày mua sắm 11/11
        "Sale_1212":         "12-12",
        "Christmas":         "12-25",
        "New_Year_Eve":      "12-31",
    }
    for yr in range(2012, 2025):
        for name, mmdd in fixed.items():
            try:
                rows.append(dict(holiday=name, ds=pd.Timestamp(f"{yr}-{mmdd}"),
                                 lower_window=-1, upper_window=1))
            except ValueError:
                pass
    return pd.DataFrame(rows)
 
vn_holidays = build_prophet_holidays()
 
# Set ngày Tết & lễ để dùng làm binary features cho LGBM
TET_WINDOW: set = set()
HOLIDAY_MMDD: set = {"01-01","04-30","05-01","09-02","11-11","12-12","12-25","12-31"}
 
for s in TET.values():
    tet = pd.Timestamp(s)
    for d in range(-7, 8):
        TET_WINDOW.add(tet + pd.Timedelta(days=d))
 
print(f"  Holidays defined: {vn_holidays['holiday'].nunique()} types, "
      f"{len(vn_holidays)} entries")
 


STEP 2 — Building Vietnam holiday calendar
  Holidays defined: 10 types, 128 entries


In [5]:
# %% ── [3] Prophet fit & component extraction ─────────────────────────────────
# KEY TRICK: Không chỉ dùng Prophet để predict, mà còn extract các COMPONENTS
# (trend, yearly, weekly, holiday effects) làm FEATURES cho LGBM.
# Đây là cách tốt nhất kết hợp Prophet + tree models.
print("\nSTEP 3 — Fitting Prophet & extracting decomposition components")
 
def fit_prophet_full(train_df: pd.DataFrame, target: str) -> Prophet:
    """Fit Prophet với full seasonality + VN holidays."""
    pdf = (train_df[["Date", target]]
           .rename(columns={"Date": "ds", target: "y"}))
    m = Prophet(
        yearly_seasonality   = 10,         # Fourier order
        weekly_seasonality   = 5,
        daily_seasonality    = False,
        holidays             = vn_holidays,
        seasonality_mode     = "multiplicative",  # phù hợp data có trend tăng
        changepoint_prior_scale    = 0.05,
        seasonality_prior_scale    = 10.0,
        holidays_prior_scale       = 10.0,
        interval_width             = 0.95,
    )
    m.add_seasonality(name="monthly",   period=30.5,  fourier_order=5)
    m.add_seasonality(name="quarterly", period=91.25, fourier_order=3)
    m.fit(pdf, seed=SEED)
    return m
 
def get_prophet_components(model: Prophet,
                           dates: pd.Series) -> pd.DataFrame:
    """
    Trả về DataFrame gồm: yhat, trend, yearly, weekly,
    monthly, quarterly, holiday effects.
    Đây là features cho LGBM — rất mạnh vì capture được
    cả trend lẫn seasonality đã được Prophet học.
    """
    future = model.make_future_dataframe(periods=0)
    future = pd.DataFrame({"ds": dates})
    comp   = model.predict(future)
    cols_want = ["ds", "yhat", "trend", "yearly", "weekly",
                 "monthly", "quarterly"]
    holiday_cols = [c for c in comp.columns
                    if c.startswith("tet") or c.startswith("Sale")
                    or c in ("New_Year","Reunification_Day","Labor_Day",
                             "National_Day","Christmas","New_Year_Eve")]
    cols_want += [c for c in holiday_cols if c in comp.columns]
    return comp[[c for c in cols_want if c in comp.columns]].rename(
        columns={"ds": "Date"})
 
 
# Fit trên tập train (2012–2021) trước — dùng để evaluate trên 2022
train_mask = df["Date"] < "2022-01-01"
val_mask   = df["Date"] >= "2022-01-01"
 
print("  Fitting Prophet for Revenue (train period)...")
p_rev_tr  = fit_prophet_full(df[train_mask], "Revenue")
print("  Fitting Prophet for COGS (train period)...")
p_cogs_tr = fit_prophet_full(df[train_mask], "COGS")
 
# Extract components cho cả train + val
comp_rev_all  = get_prophet_components(p_rev_tr,  df["Date"])
comp_cogs_all = get_prophet_components(p_cogs_tr, df["Date"])
 
# Rename columns để phân biệt rev vs cogs components
comp_rev_all  = comp_rev_all.rename( columns={c: f"p_rev_{c}"  for c in comp_rev_all.columns  if c != "Date"})
comp_cogs_all = comp_cogs_all.rename(columns={c: f"p_cogs_{c}" for c in comp_cogs_all.columns if c != "Date"})
 
df = df.merge(comp_rev_all,  on="Date", how="left")
df = df.merge(comp_cogs_all, on="Date", how="left")
print(f"  Prophet components added: "
      f"{[c for c in df.columns if c.startswith('p_rev_') or c.startswith('p_cogs_')]}")
 


STEP 3 — Fitting Prophet & extracting decomposition components
  Fitting Prophet for Revenue (train period)...


01:30:13 - cmdstanpy - INFO - Chain [1] start processing
01:30:14 - cmdstanpy - INFO - Chain [1] done processing


  Fitting Prophet for COGS (train period)...


01:30:14 - cmdstanpy - INFO - Chain [1] start processing
01:30:15 - cmdstanpy - INFO - Chain [1] done processing


  Prophet components added: ['p_rev_yhat', 'p_rev_trend', 'p_rev_yearly', 'p_rev_weekly', 'p_rev_monthly', 'p_rev_quarterly', 'p_rev_Christmas', 'p_rev_Labor_Day', 'p_rev_National_Day', 'p_rev_New_Year', 'p_rev_New_Year_Eve', 'p_rev_Reunification_Day', 'p_rev_Sale_1111', 'p_rev_Sale_1111_lower', 'p_rev_Sale_1111_upper', 'p_rev_Sale_1212', 'p_rev_Sale_1212_lower', 'p_rev_Sale_1212_upper', 'p_rev_tet_break', 'p_rev_tet_break_lower', 'p_rev_tet_break_upper', 'p_rev_tet_pre_spike', 'p_rev_tet_pre_spike_lower', 'p_rev_tet_pre_spike_upper', 'p_cogs_yhat', 'p_cogs_trend', 'p_cogs_yearly', 'p_cogs_weekly', 'p_cogs_monthly', 'p_cogs_quarterly', 'p_cogs_Christmas', 'p_cogs_Labor_Day', 'p_cogs_National_Day', 'p_cogs_New_Year', 'p_cogs_New_Year_Eve', 'p_cogs_Reunification_Day', 'p_cogs_Sale_1111', 'p_cogs_Sale_1111_lower', 'p_cogs_Sale_1111_upper', 'p_cogs_Sale_1212', 'p_cogs_Sale_1212_lower', 'p_cogs_Sale_1212_upper', 'p_cogs_tet_break', 'p_cogs_tet_break_lower', 'p_cogs_tet_break_upper', 'p_cogs

In [6]:
# %% ── [4] Feature Engineering ────────────────────────────────────────────────
print("\nSTEP 4 — Feature engineering")
 
def build_features(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo toàn bộ features từ chuỗi thời gian + Prophet components.
    Không dùng bất kỳ file nào khác ngoài sales.csv.
    """
    d = df_in.copy()
 
    # ── Calendar ──────────────────────────────────────────────
    d["year"]           = d["Date"].dt.year
    d["month"]          = d["Date"].dt.month
    d["day"]            = d["Date"].dt.day
    d["quarter"]        = d["Date"].dt.quarter
    d["dayofweek"]      = d["Date"].dt.dayofweek
    d["dayofyear"]      = d["Date"].dt.dayofyear
    d["weekofyear"]     = d["Date"].dt.isocalendar().week.astype(int)
    d["is_weekend"]     = d["dayofweek"].isin([5, 6]).astype(int)
    d["is_month_start"] = d["Date"].dt.is_month_start.astype(int)
    d["is_month_end"]   = d["Date"].dt.is_month_end.astype(int)
    d["is_quarter_end"] = d["Date"].dt.is_quarter_end.astype(int)
    d["is_year_end"]    = ((d["month"] == 12) & (d["day"] == 31)).astype(int)
 
    # ── Fourier encoding — tốt hơn raw dayofyear/month ────────
    d["sin_doy"]   = np.sin(2 * np.pi * d["dayofyear"] / 365.25)
    d["cos_doy"]   = np.cos(2 * np.pi * d["dayofyear"] / 365.25)
    d["sin_doy2"]  = np.sin(4 * np.pi * d["dayofyear"] / 365.25)  # 2nd harmonic
    d["cos_doy2"]  = np.cos(4 * np.pi * d["dayofyear"] / 365.25)
    d["sin_dow"]   = np.sin(2 * np.pi * d["dayofweek"] / 7)
    d["cos_dow"]   = np.cos(2 * np.pi * d["dayofweek"] / 7)
    d["sin_mon"]   = np.sin(2 * np.pi * d["month"] / 12)
    d["cos_mon"]   = np.cos(2 * np.pi * d["month"] / 12)
 
    # ── Holiday & Tết flags ────────────────────────────────────
    d["is_holiday"]    = d["Date"].dt.strftime("%m-%d").isin(HOLIDAY_MMDD).astype(int)
    d["is_tet_period"] = d["Date"].isin(TET_WINDOW).astype(int)
    # Days until Tết / days since Tết (continuous proximity feature)
    tet_ts = pd.Series(sorted(TET_WINDOW))
    def days_to_nearest_tet(date):
        diffs = (tet_ts - date).abs()
        return diffs.min().days
    d["days_to_tet"] = d["Date"].apply(days_to_nearest_tet)
 
    # ── Lag features ───────────────────────────────────────────
    for lag in [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 180, 365]:
        d[f"rev_lag_{lag}"]  = d["Revenue"].shift(lag)
        d[f"cogs_lag_{lag}"] = d["COGS"].shift(lag)
 
    # ── Rolling statistics (shift(1) để tránh leakage) ────────
    rev_s1  = d["Revenue"].shift(1)
    cogs_s1 = d["COGS"].shift(1)
    for w in [7, 14, 30, 60, 90]:
        d[f"rev_roll_mean_{w}"]  = rev_s1.rolling(w).mean()
        d[f"rev_roll_std_{w}"]   = rev_s1.rolling(w).std()
        d[f"rev_roll_min_{w}"]   = rev_s1.rolling(w).min()
        d[f"rev_roll_max_{w}"]   = rev_s1.rolling(w).max()
        d[f"cogs_roll_mean_{w}"] = cogs_s1.rolling(w).mean()
        d[f"cogs_roll_std_{w}"]  = cogs_s1.rolling(w).std()
 
    # ── EWMA — capture momentum tốt hơn rolling mean ──────────
    for span in [7, 14, 30, 90]:
        d[f"rev_ewm_{span}"]  = rev_s1.ewm(span=span, min_periods=3).mean()
        d[f"cogs_ewm_{span}"] = cogs_s1.ewm(span=span, min_periods=3).mean()
 
    # ── Derived / ratio features ───────────────────────────────
    d["gross_margin_lag1"] = (
        (d["Revenue"].shift(1) - d["COGS"].shift(1)) /
        (d["Revenue"].shift(1) + 1e-6)
    )
    d["cogs_ratio_lag1"] = d["COGS"].shift(1) / (d["Revenue"].shift(1) + 1e-6)
 
    # YoY: so sánh với cùng kỳ năm ngoái
    d["rev_yoy"]  = d["Revenue"].shift(1) / (d["Revenue"].shift(365) + 1e-6)
    d["cogs_yoy"] = d["COGS"].shift(1)    / (d["COGS"].shift(365) + 1e-6)
 
    # MoM: so sánh với cùng kỳ tháng trước (shift 30)
    d["rev_mom"]  = d["Revenue"].shift(1) / (d["Revenue"].shift(30) + 1e-6)
 
        # Trend features (linear regression slope ngắn hạn)
    for w in [7, 30]:
        roll_idx = np.arange(w)
        def _slope(arr):
            # arr là numpy array khi raw=True
            if len(arr) < w or np.isnan(arr).any():
                return np.nan
            return np.polyfit(roll_idx, arr, 1)[0]
        d[f"rev_trend_{w}"] = rev_s1.rolling(w).apply(_slope, raw=True)
    
    # ── Interaction features ───────────────────────────────────
    # Prophet trend × is_weekend (weekend effect on top of trend)
    if "p_rev_trend" in d.columns:
        d["trend_x_weekend"] = d["p_rev_trend"] * d["is_weekend"]
        d["trend_x_tet"]     = d["p_rev_trend"] * d["is_tet_period"]
 
    return d
 
df = build_features(df)
df = df.dropna().reset_index(drop=True)
print(f"  Total features: {df.shape[1]}  |  Rows after dropna: {len(df):,}")
 


STEP 4 — Feature engineering
  Total features: 145  |  Rows after dropna: 3,468


In [7]:
# %% ── [5] Feature column lists ───────────────────────────────────────────────
EXCLUDE = {"Date", "Revenue", "COGS"}
PROPHET_COMP_COLS = [c for c in df.columns
                     if c.startswith("p_rev_") or c.startswith("p_cogs_")]
ALL_FEATURES = [c for c in df.columns if c not in EXCLUDE]
 
# Revenue features include everything (Prophet components của cả rev & cogs)
FEAT_REV  = ALL_FEATURES
 
# COGS features — bỏ Prophet rev components để tránh circular dependency
FEAT_COGS = [c for c in ALL_FEATURES if not c.startswith("p_rev_")]
 
print(f"  Features for Revenue : {len(FEAT_REV)}")
print(f"  Features for COGS    : {len(FEAT_COGS)}")
 
 

  Features for Revenue : 142
  Features for COGS    : 118


In [8]:
# %% ── [6] Model definitions ──────────────────────────────────────────────────
def get_models():
    return {
        "lgb": LGBMRegressor(
            n_estimators       = 3000,
            learning_rate      = 0.01,
            num_leaves         = 63,
            max_depth          = -1,
            min_child_samples  = 20,
            subsample          = 0.75,
            colsample_bytree   = 0.75,
            reg_alpha          = 0.1,
            reg_lambda         = 1.0,
            random_state       = SEED,
            verbose            = -1,
            n_jobs             = -1,
        ),
        "xgb": xgb.XGBRegressor(
            n_estimators       = 3000,
            learning_rate      = 0.01,
            max_depth          = 6,
            min_child_weight   = 5,
            subsample          = 0.75,
            colsample_bytree   = 0.75,
            reg_alpha          = 0.1,
            reg_lambda         = 1.0,
            random_state       = SEED,
            tree_method        = "hist",
            verbosity          = 0,
            n_jobs             = -1,
        ),
        "cat": CatBoostRegressor(
            iterations         = 2000,
            learning_rate      = 0.02,
            depth              = 6,
            l2_leaf_reg        = 3,
            random_seed        = SEED,
            verbose            = 0,
        ),
    }
 

In [9]:
# %% ── [7] OOF Stacking (TimeSeriesSplit — không leakage) ────────────────────
def metrics(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    if label:
        print(f"  {label:<40} MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R²={r2:.4f}")
    return mae, rmse, r2
 
def oof_stacking(X: pd.DataFrame, y: pd.Series,
                 models_dict: dict, n_splits=5, gap=30):
    """
    TimeSeriesSplit OOF: mỗi fold train set luôn đứng trước val set.
    gap=30: bỏ 30 ngày giữa train và val để lag features không bị leakage.
    Trả về oof predictions shape (N, n_models).
    """
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    oof  = np.zeros((len(X), len(models_dict)))
    names = list(models_dict.keys())
 
    print(f"    TimeSeriesSplit: {n_splits} folds, gap={gap} days")
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        fold_mae = []
        for i, (name, m) in enumerate(models_dict.items()):
            clf = copy.deepcopy(m)
            clf.fit(X_tr, y_tr)
            pred = clf.predict(X_val)
            oof[val_idx, i] = pred
            fold_mae.append(mean_absolute_error(y_val, pred))
        fold_str = " | ".join(f"{n}={v:,.0f}" for n, v in zip(names, fold_mae))
        print(f"    Fold {fold+1}: MAE → {fold_str}")
 
    return oof
 
def fit_final(X: pd.DataFrame, y: pd.Series,
              models_dict: dict, oof_preds: np.ndarray):
    """
    1. Fit meta-learner (Ridge) trên OOF predictions → no leakage
    2. Retrain tất cả base models trên full (X, y)
    """
    meta = Ridge(alpha=1.0, positive=True)
    meta.fit(oof_preds, y)
    weights = {n: round(w, 4) for n, w in zip(models_dict.keys(), meta.coef_)}
    print(f"    Meta-learner weights: {weights}")
 
    fitted = {}
    for name, m in models_dict.items():
        clf = copy.deepcopy(m)
        clf.fit(X, y)
        fitted[name] = clf
    return fitted, meta
 
def stack_predict(X, fitted_base, meta, prophet_yhat):
    """Prophet yhat + meta-weighted residual dari base models."""
    base_preds = np.column_stack([m.predict(X) for m in fitted_base.values()])
    residual   = meta.predict(base_preds)
    return np.maximum(prophet_yhat + residual, 0.0)
 
 

In [10]:
# %% ── [8] Validation run (train=2012-2021, val=2022) ────────────────────────
print("\n" + "=" * 65)
print("STEP 8 — Validation (train ≤2021, val = 2022)")
print("=" * 65)
 
train_mask = df["Date"] < "2022-01-01"
val_mask   = df["Date"] >= "2022-01-01"
 
# Residuals = Actual - Prophet yhat (what LGBM/XGB/CAT will learn)
df["res_rev"]  = df["Revenue"] - df["p_rev_yhat"]
df["res_cogs"] = df["COGS"]    - df["p_cogs_yhat"]
 
X_tr  = df.loc[train_mask, FEAT_REV];  y_tr  = df.loc[train_mask, "res_rev"]
X_val = df.loc[val_mask,   FEAT_REV];  y_val = df.loc[val_mask,   "res_rev"]
 
X_tr_c  = df.loc[train_mask, FEAT_COGS]; y_tr_c  = df.loc[train_mask, "res_cogs"]
X_val_c = df.loc[val_mask,   FEAT_COGS]; y_val_c = df.loc[val_mask,   "res_cogs"]
 
# OOF on train portion
print("\n[Revenue — Residuals]")
models_rev_val = get_models()
oof_rev = oof_stacking(X_tr, y_tr, models_rev_val, n_splits=5, gap=30)
fitted_rev_val, meta_rev_val = fit_final(X_tr, y_tr, models_rev_val, oof_rev)
 
print("\n[COGS — Residuals]")
models_cogs_val = get_models()
oof_cogs = oof_stacking(X_tr_c, y_tr_c, models_cogs_val, n_splits=5, gap=30)
fitted_cogs_val, meta_cogs_val = fit_final(X_tr_c, y_tr_c, models_cogs_val, oof_cogs)
 
# Validate on 2022
val_rev_pred  = stack_predict(X_val,   fitted_rev_val,  meta_rev_val,
                               df.loc[val_mask, "p_rev_yhat"].values)
val_cogs_pred = stack_predict(X_val_c, fitted_cogs_val, meta_cogs_val,
                               df.loc[val_mask, "p_cogs_yhat"].values)
 
print("\n── Validation results (2022) ──")
metrics(df.loc[val_mask, "Revenue"], val_rev_pred,  "Revenue: Prophet+Stack Ensemble")
metrics(df.loc[val_mask, "Revenue"],
        df.loc[val_mask, "p_rev_yhat"].values,      "Revenue: Prophet only (baseline)")
print()
metrics(df.loc[val_mask, "COGS"],   val_cogs_pred,  "COGS: Prophet+Stack Ensemble")
metrics(df.loc[val_mask, "COGS"],
        df.loc[val_mask, "p_cogs_yhat"].values,     "COGS: Prophet only (baseline)")
 


STEP 8 — Validation (train ≤2021, val = 2022)

[Revenue — Residuals]
    TimeSeriesSplit: 5 folds, gap=30 days
    Fold 1: MAE → lgb=963,796 | xgb=942,116 | cat=960,097
    Fold 2: MAE → lgb=1,076,145 | xgb=1,076,653 | cat=1,067,994
    Fold 3: MAE → lgb=787,946 | xgb=787,320 | cat=801,689
    Fold 4: MAE → lgb=673,107 | xgb=603,616 | cat=620,604
    Fold 5: MAE → lgb=494,263 | xgb=478,927 | cat=491,313
    Meta-learner weights: {'lgb': np.float64(0.2181), 'xgb': np.float64(0.3292), 'cat': np.float64(0.4824)}


KeyboardInterrupt: 

In [ ]:
# %% ── [9] SHAP Explainability ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 9 — SHAP Feature Importance (cho báo cáo kỹ thuật)")
print("=" * 65)
 
lgb_model = fitted_rev_val["lgb"]
explainer  = shap.TreeExplainer(lgb_model)
shap_vals  = explainer.shap_values(X_val)
 
shap_df = (pd.DataFrame({"feature": FEAT_REV,
                          "mean_abs_shap": np.abs(shap_vals).mean(axis=0)})
           .sort_values("mean_abs_shap", ascending=False)
           .reset_index(drop=True))
 
print("\nTop 20 features — Revenue model:")
print(shap_df.head(20).to_string(index=False))
 


STEP 9 — SHAP Feature Importance (cho báo cáo kỹ thuật)

Top 20 features — Revenue model:
         feature  mean_abs_shap
     p_cogs_yhat  225224.687677
       rev_lag_1  206014.290612
      p_rev_yhat  148305.670620
             day  124344.888466
    cogs_lag_365   99019.075091
   p_rev_monthly   75064.787605
       rev_lag_7   51067.204070
     cogs_lag_14   50067.686475
      cogs_lag_7   46513.221197
     p_rev_trend   42638.328741
    rev_trend_30   40319.208915
         rev_mom   39191.089367
         sin_doy   36709.851467
     rev_trend_7   35366.768231
  p_cogs_monthly   32460.406318
        sin_doy2   31042.193861
 p_rev_quarterly   28274.735075
        cogs_yoy   27754.159180
p_cogs_quarterly   27435.636588
      rev_lag_14   27095.959121


In [ ]:
# Business interpretation tự động
BUSINESS_NOTE = {
    "p_rev_yhat"   : "Prophet dự báo tổng thể (trend + seasonality)",
    "p_rev_trend"  : "Xu hướng tăng trưởng dài hạn",
    "p_rev_yearly" : "Seasonality hàng năm (peak cuối năm, Tết...)",
    "p_rev_weekly" : "Hành vi mua sắm theo ngày trong tuần",
    "is_tet_period": "Tết Nguyên Đán tác động mạnh đến doanh thu",
    "days_to_tet"  : "Khoảng cách đến Tết → spike mua sắm trước Tết",
    "rev_ewm_7"    : "Momentum ngắn hạn (7 ngày) phản ánh campaign hiện tại",
    "rev_ewm_30"   : "Momentum trung hạn (30 ngày) — trend tháng",
    "rev_yoy"      : "So sánh cùng kỳ năm ngoái — growth rate",
    "is_month_end" : "Chi tiêu cuối tháng — hành vi consumer",
    "year"         : "Tăng trưởng dài hạn của doanh nghiệp theo năm",
    "rev_roll_mean_30": "Trung bình 30 ngày — baseline tháng hiện tại",
}
 
print("\n── Business interpretation (top 10) ──")
for _, row in shap_df.head(10).iterrows():
    note = BUSINESS_NOTE.get(row["feature"], "Feature ảnh hưởng đáng kể đến dự báo")
    print(f"  {row['feature']:<35} SHAP={row['mean_abs_shap']:>10,.0f}  →  {note}")
 
# Save SHAP summary plot
fig_shap, ax_shap = plt.subplots(figsize=(9, 7))
top15 = shap_df.head(15)
ax_shap.barh(top15["feature"][::-1], top15["mean_abs_shap"][::-1],
             color="#378ADD", edgecolor="none")
ax_shap.set_xlabel("Mean |SHAP value|", fontsize=11)
ax_shap.set_title("Feature Importance — Revenue (SHAP)\nValidation set 2022",
                  fontsize=12, fontweight="bold")
ax_shap.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "shap_revenue.png"), dpi=150)
plt.close()
print("\n  Saved: outputs/shap_revenue.png")
 


── Business interpretation (top 10) ──
  p_cogs_yhat                         SHAP=   225,225  →  Feature ảnh hưởng đáng kể đến dự báo
  rev_lag_1                           SHAP=   206,014  →  Feature ảnh hưởng đáng kể đến dự báo
  p_rev_yhat                          SHAP=   148,306  →  Prophet dự báo tổng thể (trend + seasonality)
  day                                 SHAP=   124,345  →  Feature ảnh hưởng đáng kể đến dự báo
  cogs_lag_365                        SHAP=    99,019  →  Feature ảnh hưởng đáng kể đến dự báo
  p_rev_monthly                       SHAP=    75,065  →  Feature ảnh hưởng đáng kể đến dự báo
  rev_lag_7                           SHAP=    51,067  →  Feature ảnh hưởng đáng kể đến dự báo
  cogs_lag_14                         SHAP=    50,068  →  Feature ảnh hưởng đáng kể đến dự báo
  cogs_lag_7                          SHAP=    46,513  →  Feature ảnh hưởng đáng kể đến dự báo
  p_rev_trend                         SHAP=    42,638  →  Xu hướng tăng trưởng dài hạn

  Saved:

In [ ]:
# %% ── [10] Retrain on FULL data (2012–2022) ──────────────────────────────────
print("\n" + "=" * 65)
print("STEP 10 — Retrain on full data (2012–2022)")
print("=" * 65)
 
# Refit Prophet trên toàn bộ train data
print("  Refitting Prophet on full data...")
p_rev_full  = fit_prophet_full(df, "Revenue")
p_cogs_full = fit_prophet_full(df, "COGS")
 
comp_rev_full  = get_prophet_components(p_rev_full,  df["Date"])
comp_cogs_full = get_prophet_components(p_cogs_full, df["Date"])
comp_rev_full  = comp_rev_full.rename( columns={c: f"p_rev_{c}"  for c in comp_rev_full.columns  if c != "Date"})
comp_cogs_full = comp_cogs_full.rename(columns={c: f"p_cogs_{c}" for c in comp_cogs_full.columns if c != "Date"})
 
# Update dataframe với components mới (trained on full data)
df_full = df.drop(columns=[c for c in df.columns
                            if c.startswith("p_rev_") or c.startswith("p_cogs_")])
df_full = df_full.merge(comp_rev_full,  on="Date", how="left")
df_full = df_full.merge(comp_cogs_full, on="Date", how="left")
df_full["res_rev"]  = df_full["Revenue"] - df_full["p_rev_yhat"]
df_full["res_cogs"] = df_full["COGS"]    - df_full["p_cogs_yhat"]
 
# Recompute features với full data
df_full = build_features(df_full.drop(columns=[c for c in df_full.columns
    if c not in {"Date","Revenue","COGS"}
    and not c.startswith("p_rev_")
    and not c.startswith("p_cogs_")]))
df_full = df_full.dropna().reset_index(drop=True)
df_full["res_rev"]  = df_full["Revenue"] - df_full["p_rev_yhat"]
df_full["res_cogs"] = df_full["COGS"]    - df_full["p_cogs_yhat"]
 
X_full_rev  = df_full[FEAT_REV]
y_full_rev  = df_full["res_rev"]
X_full_cogs = df_full[FEAT_COGS]
y_full_cogs = df_full["res_cogs"]
 
print("\n[Revenue — Full OOF]")
models_rev_final = get_models()
oof_rev_full = oof_stacking(X_full_rev, y_full_rev, models_rev_final, n_splits=5, gap=30)
fitted_rev_final, meta_rev_final = fit_final(X_full_rev, y_full_rev,
                                              models_rev_final, oof_rev_full)
 
print("\n[COGS — Full OOF]")
models_cogs_final = get_models()
oof_cogs_full = oof_stacking(X_full_cogs, y_full_cogs, models_cogs_final, n_splits=5, gap=30)
fitted_cogs_final, meta_cogs_final = fit_final(X_full_cogs, y_full_cogs,
                                                models_cogs_final, oof_cogs_full)
print("\n  Full retrain complete.")
 


STEP 10 — Retrain on full data (2012–2022)
  Refitting Prophet on full data...


01:01:31 - cmdstanpy - INFO - Chain [1] start processing
01:01:32 - cmdstanpy - INFO - Chain [1] done processing
01:01:33 - cmdstanpy - INFO - Chain [1] start processing
01:01:33 - cmdstanpy - INFO - Chain [1] done processing



[Revenue — Full OOF]
    TimeSeriesSplit: 5 folds, gap=30 days
    Fold 1: MAE → lgb=1,042,691 | xgb=1,057,751 | cat=1,060,747
    Fold 2: MAE → lgb=949,675 | xgb=965,677 | cat=983,344
    Fold 3: MAE → lgb=648,640 | xgb=657,702 | cat=610,380
    Fold 4: MAE → lgb=654,832 | xgb=632,199 | cat=655,253
    Fold 5: MAE → lgb=538,614 | xgb=531,165 | cat=542,338
    Meta-learner weights: {'lgb': np.float64(0.5254), 'xgb': np.float64(0.0), 'cat': np.float64(0.4889)}

[COGS — Full OOF]
    TimeSeriesSplit: 5 folds, gap=30 days
    Fold 1: MAE → lgb=851,899 | xgb=884,587 | cat=871,535
    Fold 2: MAE → lgb=830,903 | xgb=854,237 | cat=862,432
    Fold 3: MAE → lgb=572,691 | xgb=561,372 | cat=596,264
    Fold 4: MAE → lgb=548,001 | xgb=546,484 | cat=557,032
    Fold 5: MAE → lgb=495,578 | xgb=494,036 | cat=494,069
    Meta-learner weights: {'lgb': np.float64(0.6818), 'xgb': np.float64(0.0), 'cat': np.float64(0.269)}

  Full retrain complete.


In [ ]:
# %% ── [11] Recursive Forecast — Test Period ──────────────────────────────────
print("\n" + "=" * 65)
print("STEP 11 — Recursive Forecast (2023-01-01 → 2024-07-01)")
print("=" * 65)
 
# Prophet predict toàn bộ test dates trước (không cần recursive cho Prophet)
test_dates = sub["Date"]
future_all = pd.DataFrame({"ds": test_dates})
prophet_test_rev  = p_rev_full.predict(future_all)["yhat"].values
prophet_test_cogs = p_cogs_full.predict(future_all)["yhat"].values
 
# Lấy Prophet components cho test (để dùng làm features)
comp_test_rev  = get_prophet_components(p_rev_full,  test_dates)
comp_test_cogs = get_prophet_components(p_cogs_full, test_dates)
 
# Historical buffer (toàn bộ train) để build lag features
hist = df_full[["Date","Revenue","COGS"]].copy().reset_index(drop=True)
 
def build_single_day_features(date, hist_df,
                               p_rev_comps, p_cogs_comps) -> dict:
    """
    Tạo feature vector cho 1 ngày trong test period.
    Lag features dùng hist_df (được update sau mỗi bước recursive).
    """
    rev_s = hist_df["Revenue"]
    cog_s = hist_df["COGS"]
 
    f = {}
 
    # Calendar
    f["year"]           = date.year
    f["month"]          = date.month
    f["day"]            = date.day
    f["quarter"]        = (date.month - 1) // 3 + 1
    f["dayofweek"]      = date.dayofweek
    f["dayofyear"]      = date.timetuple().tm_yday
    f["weekofyear"]     = date.isocalendar()[1]
    f["is_weekend"]     = int(date.dayofweek in [5, 6])
    f["is_month_start"] = int(date.day == 1)
    f["is_month_end"]   = int((date + pd.Timedelta(days=1)).month != date.month)
    f["is_quarter_end"] = int(f["is_month_end"] and date.month in [3, 6, 9, 12])
    f["is_year_end"]    = int(date.month == 12 and date.day == 31)
 
    doy = f["dayofyear"]; dow = f["dayofweek"]; mon = f["month"]
    f["sin_doy"]  = np.sin(2 * np.pi * doy / 365.25)
    f["cos_doy"]  = np.cos(2 * np.pi * doy / 365.25)
    f["sin_doy2"] = np.sin(4 * np.pi * doy / 365.25)
    f["cos_doy2"] = np.cos(4 * np.pi * doy / 365.25)
    f["sin_dow"]  = np.sin(2 * np.pi * dow / 7)
    f["cos_dow"]  = np.cos(2 * np.pi * dow / 7)
    f["sin_mon"]  = np.sin(2 * np.pi * mon / 12)
    f["cos_mon"]  = np.cos(2 * np.pi * mon / 12)
 
    f["is_holiday"]    = int(date.strftime("%m-%d") in HOLIDAY_MMDD)
    f["is_tet_period"] = int(date in TET_WINDOW)
    tet_ts = pd.Series(sorted(TET_WINDOW))
    f["days_to_tet"]   = int((tet_ts - date).abs().min().days)
 
    # Lag revenue
    for lag in [1,2,3,7,14,21,28,30,60,90,180,365]:
        idx = len(rev_s) - lag
        f[f"rev_lag_{lag}"]  = float(rev_s.iloc[idx])  if idx >= 0 else float(rev_s.mean())
        f[f"cogs_lag_{lag}"] = float(cog_s.iloc[idx])  if idx >= 0 else float(cog_s.mean())
 
    # Rolling revenue
    for w in [7,14,30,60,90]:
        tail_r = rev_s.iloc[-w:] if len(rev_s) >= w else rev_s
        tail_c = cog_s.iloc[-w:] if len(cog_s) >= w else cog_s
        f[f"rev_roll_mean_{w}"]  = float(tail_r.mean())
        f[f"rev_roll_std_{w}"]   = float(tail_r.std()) if len(tail_r) > 1 else 0.0
        f[f"rev_roll_min_{w}"]   = float(tail_r.min())
        f[f"rev_roll_max_{w}"]   = float(tail_r.max())
        f[f"cogs_roll_mean_{w}"] = float(tail_c.mean())
        f[f"cogs_roll_std_{w}"]  = float(tail_c.std()) if len(tail_c) > 1 else 0.0
 
    # EWMA
    for span in [7,14,30,90]:
        f[f"rev_ewm_{span}"]  = float(rev_s.ewm(span=span).mean().iloc[-1])
        f[f"cogs_ewm_{span}"] = float(cog_s.ewm(span=span).mean().iloc[-1])
 
    # Derived
    r1 = float(rev_s.iloc[-1]); c1 = float(cog_s.iloc[-1])
    f["gross_margin_lag1"] = (r1 - c1) / (r1 + 1e-6)
    f["cogs_ratio_lag1"]   = c1 / (r1 + 1e-6)
    r365 = float(rev_s.iloc[-365]) if len(rev_s) >= 365 else float(rev_s.mean())
    c365 = float(cog_s.iloc[-365]) if len(cog_s) >= 365 else float(cog_s.mean())
    r30  = float(rev_s.iloc[-30])  if len(rev_s) >= 30  else float(rev_s.mean())
    f["rev_yoy"]  = r1 / (r365 + 1e-6)
    f["cogs_yoy"] = c1 / (c365 + 1e-6)
    f["rev_mom"]  = r1 / (r30  + 1e-6)
 
    # Trend slope
    for w in [7, 30]:
        tail = rev_s.iloc[-w:].values if len(rev_s) >= w else rev_s.values
        if len(tail) >= 3:
            f[f"rev_trend_{w}"] = float(np.polyfit(np.arange(len(tail)), tail, 1)[0])
        else:
            f[f"rev_trend_{w}"] = 0.0
 
    # Prophet components (đã precomputed)
    day_rev  = p_rev_comps[p_rev_comps["Date"] == date]
    day_cogs = p_cogs_comps[p_cogs_comps["Date"] == date]
    for col in [c for c in p_rev_comps.columns if c != "Date"]:
        f[col] = float(day_rev[col].values[0]) if len(day_rev) > 0 else 0.0
    for col in [c for c in p_cogs_comps.columns if c != "Date"]:
        f[col] = float(day_cogs[col].values[0]) if len(day_cogs) > 0 else 0.0
 
    # Interaction
    if "p_rev_trend" in f:
        f["trend_x_weekend"] = f["p_rev_trend"] * f["is_weekend"]
        f["trend_x_tet"]     = f["p_rev_trend"] * f["is_tet_period"]
 
    return f
 
# Prophet components cho test (renamed để match)
comp_test_rev_r  = comp_test_rev.rename( columns={c: f"p_rev_{c}"  for c in comp_test_rev.columns  if c != "Date"})
comp_test_cogs_r = comp_test_cogs.rename(columns={c: f"p_cogs_{c}" for c in comp_test_cogs.columns if c != "Date"})
 
predictions = []
print(f"  Forecasting {len(test_dates)} days recursively...")
 
for i, date in enumerate(test_dates):
    if i % 100 == 0:
        print(f"  ... day {i+1}/{len(test_dates)} ({date.date()})")
 
    feat_dict = build_single_day_features(date, hist,
                                          comp_test_rev_r, comp_test_cogs_r)
 
    # Predict COGS first (Revenue không phụ thuộc COGS ngay hôm đó)
    X_cogs_d = pd.DataFrame([feat_dict]).reindex(columns=FEAT_COGS, fill_value=0.0)
    p_cogs_d = prophet_test_cogs[i]
    base_c   = np.column_stack([m.predict(X_cogs_d) for m in fitted_cogs_final.values()])
    pred_cogs = max(0.0, float(p_cogs_d + meta_cogs_final.predict(base_c)[0]))
 
    # Predict Revenue
    X_rev_d = pd.DataFrame([feat_dict]).reindex(columns=FEAT_REV, fill_value=0.0)
    p_rev_d = prophet_test_rev[i]
    base_r  = np.column_stack([m.predict(X_rev_d) for m in fitted_rev_final.values()])
    pred_rev = max(0.0, float(p_rev_d + meta_rev_final.predict(base_r)[0]))
 
    predictions.append({"Date": date, "Revenue": pred_rev, "COGS": pred_cogs})
 
    # Update history với predicted values (recursive)
    new_row = pd.DataFrame([{"Date": date, "Revenue": pred_rev, "COGS": pred_cogs}])
    hist = pd.concat([hist, new_row], ignore_index=True)
 
print("  Recursive forecast done.")
 


STEP 11 — Recursive Forecast (2023-01-01 → 2024-07-01)
  Forecasting 548 days recursively...
  ... day 1/548 (2023-01-01)
  ... day 101/548 (2023-04-11)
  ... day 201/548 (2023-07-20)
  ... day 301/548 (2023-10-28)
  ... day 401/548 (2024-02-05)
  ... day 501/548 (2024-05-15)
  Recursive forecast done.


In [ ]:
# %% ── [12] Create submission ─────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 12 — Creating submission file")
print("=" * 65)

pred_df = pd.DataFrame(predictions)

# Đảm bảo Date trong pred_df là datetime
pred_df["Date"] = pd.to_datetime(pred_df["Date"])

# Đảm bảo sub["Date"] là datetime (đọc lại nếu cần)
# Cách 1: Chuyển đổi an toàn
sub["Date"] = pd.to_datetime(sub["Date"])

# Format date cho cả hai dataframe trước khi merge
pred_df["Date_str"] = pred_df["Date"].dt.strftime("%Y-%m-%d")
sub["Date_str"] = sub["Date"].dt.strftime("%Y-%m-%d")

# Merge theo Date_str
submission = sub[["Date_str"]].merge(
    pred_df[["Date_str", "Revenue", "COGS"]], 
    on="Date_str", 
    how="left"
)
submission = submission.rename(columns={"Date_str": "Date"})
submission = submission[["Date", "Revenue", "COGS"]]

assert len(submission) == len(sub), "Row count mismatch!"
assert submission["Revenue"].isna().sum() == 0, "NaN in Revenue!"
assert submission["COGS"].isna().sum() == 0, "NaN in COGS!"

out_path = os.path.join(OUT_DIR, "submission_khai_2.csv")
submission.to_csv(out_path, index=False)
print(f"  Saved: {out_path}")
print(f"  Shape: {submission.shape}")
print(f"\n  Preview:")
print(submission.head(10).to_string(index=False))
print(f"\n  Revenue — mean: {submission['Revenue'].mean():,.0f} | "
      f"min: {submission['Revenue'].min():,.0f} | max: {submission['Revenue'].max():,.0f}")


STEP 12 — Creating submission file


NameError: name 'pd' is not defined

In [ ]:
# %% ── [13] Visualizations ────────────────────────────────────────────────────
print("\nSTEP 13 — Saving plots...")
 
fig, axes = plt.subplots(3, 1, figsize=(16, 15), facecolor="white")
fig.suptitle("DATATHON 2026 — Revenue Forecasting Results", fontsize=14, fontweight="bold")
 
# (a) Full history + forecast
ax = axes[0]
ax.plot(df_full["Date"], df_full["Revenue"],
        color="#4477AA", linewidth=0.7, alpha=0.8, label="Actual Train (2012–2022)")
ax.plot(pd.to_datetime(submission["Date"]), submission["Revenue"],
        color="#EE6677", linewidth=1.5, label="Forecast (2023–2024)")
ax.axvline(pd.Timestamp("2023-01-01"), color="gray",
           linestyle="--", linewidth=1, alpha=0.7, label="Train/Test split")
ax.set_title("Revenue — Full Timeline", fontweight="bold")
ax.set_ylabel("Revenue (VND)")
ax.legend(fontsize=9); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
 
# (b) Validation 2022 — Actual vs Predicted
ax = axes[1]
val_actual = df.loc[val_mask, ["Date","Revenue"]].reset_index(drop=True)
ax.plot(val_actual["Date"], val_actual["Revenue"],
        color="#4477AA", linewidth=1.2, label="Actual 2022")
ax.plot(val_actual["Date"], val_rev_pred,
        color="#EE6677", linewidth=1.5, linestyle="--", label="Predicted 2022")
mae_v, rmse_v, r2_v = metrics(val_actual["Revenue"], val_rev_pred)
ax.set_title(f"Validation 2022 — MAE={mae_v:,.0f}  RMSE={rmse_v:,.0f}  R²={r2_v:.4f}",
             fontweight="bold")
ax.set_ylabel("Revenue (VND)")
ax.legend(fontsize=9)
 
# (c) Residuals validation
ax = axes[2]
res_plot = val_actual["Revenue"].values - val_rev_pred
ax.scatter(val_actual["Date"], res_plot, s=8, alpha=0.5, color="#AA3377")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Residuals — Validation 2022 (Actual − Predicted)", fontweight="bold")
ax.set_ylabel("Residual")
ax.set_xlabel("Date")
 
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "forecast_results.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: outputs/forecast_results.png")
print("\n" + "=" * 65)
print("ALL DONE! Outputs:")
print(f"  outputs/submission.csv")
print(f"  outputs/shap_revenue.png")
print(f"  outputs/forecast_results.png")
print("=" * 65)
 


STEP 13 — Saving plots...
  Saved: outputs/forecast_results.png

ALL DONE! Outputs:
  outputs/submission.csv
  outputs/shap_revenue.png
  outputs/forecast_results.png
